In [3]:
import pandas as pd
import numpy as np
import faiss

from sentence_transformers import SentenceTransformer

c:\Users\arote\anaconda3\envs\animal_fixed\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
df = pd.read_parquet(
    "../data/processed/summaries_only.parquet"
)

embeddings = np.load(
    "../data/embeddings/embeddings.npy"
)

index = faiss.read_index(
    "../models/retrieval/faiss_index.bin"
)

print("Data Shape:", df.shape)
print("Embeddings Shape:", embeddings.shape)
print("FAISS Vectors:", index.ntotal)

Data Shape: (26688, 3)
Embeddings Shape: (26688, 384)
FAISS Vectors: 26688


In [5]:
model = SentenceTransformer(
    "BAAI/bge-small-en-v1.5"
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2144.46it/s]


In [6]:
def search_cases(query, k=5):

    query_embedding = model.encode(
        [query],
        normalize_embeddings=True
    )

    scores, indices = index.search(
        query_embedding.astype("float32"),
        k
    )

    return scores[0], indices[0]

In [7]:
def show_results(query, k=5):

    scores, indices = search_cases(query, k)

    print("\n")
    print("="*80)
    print("QUERY:", query)
    print("="*80)

    for rank, (score, idx) in enumerate(
        zip(scores, indices),
        start=1
    ):

        print("\n")
        print(rank, df.iloc[idx]["case_name"])

        print(
            "Score:",
            round(float(score), 4)
        )

        print("\nSUMMARY:\n")

        print(
            df.iloc[idx]["legal_summary"][:1000]
        )

        print("\n" + "-"*80)

In [8]:
show_results(
    "agent commission dispute property sale",
    k=5
)



QUERY: agent commission dispute property sale


1 C_I_T_Mumbai_vs_M_S_Sarkar_Builders_on_15_May_2015_1
Score: 0.7569

SUMMARY:

s of the various High Courts have spoken in one voice which are questioned on identical grounds by the appellant Revenue, all these appeals were heard analogously and by this judgment, we propose to answer the question of law involved and as formulated above in order to give quietus to this surging debate. Before we come to the grip of the aforesaid central issue, it would be of some relevance to mention certain other disputes which had arisen between the Revenue and the assessees/developers of the housing projects concerning interpretation of sub-section (10) of Section 80IB. That dispute primarily related to the meaning that is to be assigned to 'housing projects' prior to 01.04.2005 because of the reason that there was no clause (d) earlier and there is no express provision in this sub-section dealing with the consequence of having a commercial establishm

In [9]:
show_results(
    "right to life and liberty",
    k=5
)



QUERY: right to life and liberty


1 People_S_Union_Of_Civil_Liberties_vs_Union_Of_India_Uoi_And_Anr_on_18_December_1996
Score: 0.7903

SUMMARY:

 of this Court in Kharak Singh v. The State of U.P. and Ors. . The question for consideration before this Court was whether "surveillance" under Chapter XX of the U.P. Police Regulations constituted an infringement of any of the fundamental rights guaranteed by Part III of the Constitution. Regulation 236(b) which permitted surveillance by "domiciliary visits at night" was held to be violative of Article 21 on the ground that there was no "law" under which the said regulation could be justified. 13. The word "life" and the expression "personal liberty" in Article 21 were elaborately considered by this Court in Kharak Singh's case. The majority read "right to privacy" as part of the right to life under Article 21 of the Constitution on the following reasoning: We have already extracted a passage from the judgment of Field, J. in Munn v. Illi

In [10]:
queries = [

    "agent commission dispute property sale",

    "freedom of speech restriction",

    "right to life and liberty",

    "property ownership dispute",

    "specific performance contract",

    "arbitration agreement dispute",

    "mortgage redemption",

    "criminal appeal murder conviction",

    "tax assessment dispute",

    "fundamental rights violation",

    "land acquisition compensation",

    "bail application",

    "illegal detention",

    "freedom of religion",

    "election dispute",

    "service matter promotion",

    "industrial dispute",

    "consumer protection",

    "constitutional validity",

    "public interest litigation"

]

In [11]:
for query in queries:

    scores, indices = search_cases(
        query,
        k=5
    )

    print("\n")
    print("="*80)
    print("QUERY:", query)
    print("="*80)

    for rank, idx in enumerate(
        indices,
        start=1
    ):

        print(
            rank,
            df.iloc[idx]["case_name"]
        )



QUERY: agent commission dispute property sale
1 C_I_T_Mumbai_vs_M_S_Sarkar_Builders_on_15_May_2015_1
2 R_C_Chandiok_Anr_vs_Chuni_Lal_Sabharwal_Ors_on_12_October_1970_1
3 Surja_vs_Hardeva_And_Ors_on_17_October_1968_1
4 Bagal_Kot_Cement_Co_vs_State_Of_Mysore_on_27_November_1975_1
5 Basavantappa_vs_Gangadhar_Narayan_Dharwadkar_Anr_on_10_September_1986_1


QUERY: freedom of speech restriction
1 Romesh_Thappar_vs_The_State_Of_Madras_on_26_May_1950_1
2 The_State_Of_Bihar_vs_Shailabala_Devi_on_26_May_1952_1
3 Sahara_India_Real_Estate_Corp_Ltd_Ors_vs_Securities_Exch_Board_Of_India_Anr_on_11_September_2012_1
4 Shreya_Singhal_vs_U_O_I_on_24_March_2015_1
5 Abp_Pvt_Ltd_Anr_vs_Union_Of_India_Ors_on_7_February_2014_1


QUERY: right to life and liberty
1 People_S_Union_Of_Civil_Liberties_vs_Union_Of_India_Uoi_And_Anr_on_18_December_1996
2 Anuradha_Bhasin_vs_Union_Of_India_on_10_January_2020_1
3 Rajeev_Kumar_Upadhyay_vs_Srikant_Upadhyay_on_19_December_2024_1
4 Shakti_Vahini_vs_Union_Of_India_on_27_M

In [12]:
case_id = 0

query = df.iloc[case_id]["legal_summary"]

scores, indices = search_cases(
    query,
    k=10
)

print(
    "Original:",
    df.iloc[case_id]["case_name"]
)

print()

for rank, idx in enumerate(
    indices,
    start=1
):

    print(
        rank,
        df.iloc[idx]["case_name"],
        round(scores[rank-1], 4)
    )

Original: A_K_Gopalan_vs_The_State_Of_Madras_Union_Of_India__on_19_May_1950_1

1 A_K_Gopalan_vs_The_State_Of_Madras_Union_Of_India__on_19_May_1950_1 1.0
2 Puranlal_Lakhanpal_vs_Union_Of_India_on_24_May_1957_1 0.8673
3 P_L_Lakhanpal_vs_The_State_Of_Jammu_And_Kashmir_on_20_December_1955_1 0.8563
4 Dr_Ram_Krishan_Bhardwaj_vs_The_State_Of_Delhi_And_Others_on_16_April_1953_1 0.8471
5 S_Krishnan_And_Others_vs_The_State_Of_Madras_And_Other_on_7_May_1951_1 0.8461
6 State_Of_West_Bengal_vs_Ashok_Dey_Ors_Etc_Etc_on_19_November_1971_1 0.8362
7 Hans_Muller_Of_Nurenburg_vs_Superintendent_Presidency_on_23_February_1955_1 0.834
8 Ram_Singh_vs_The_State_Of_Delhi_And_Anotherbalraj_on_6_April_1951_1 0.8294
9 Abdul_Ghani_vs_State_Of_Jammu_Kashmir_on_18_December_1970_1 0.8247
10 Khaidem_Ibocha_Singh_Etc_vs_State_Of_Manipur_on_8_October_1971_1 0.8247


In [13]:
abdulla_idx = df[
    df["case_name"].str.contains(
        "Abdulla",
        case=False,
        na=False
    )
].index[0]

query = df.iloc[
    abdulla_idx
]["legal_summary"]

scores, indices = search_cases(
    query,
    k=10
)

print(
    "Query Case:"
)

print(
    df.iloc[abdulla_idx]["case_name"]
)

print()

for rank, idx in enumerate(
    indices,
    start=1
):

    print(
        rank,
        df.iloc[idx]["case_name"],
        round(scores[rank-1], 4)
    )

Query Case:
Abdulla_Ahmed_vs_Animendra_Kissen_Mitter_on_14_March_1950_1

1 Abdulla_Ahmed_vs_Animendra_Kissen_Mitter_on_14_March_1950_1 1.0
2 Jawaharlal_Wadhwa_And_Another_vs_Haripada_Chakroberty_on_14_October_1988_1 0.8506
3 A_Kanthamani_vs_Nasreen_Ahmed_on_6_March_2017_1 0.845
4 Badrilal_vs_Municipal_Corporation_Of_Indore_on_6_December_1972_1 0.8374
5 Shah_Dhansukhlal_Chhaganlal_vs_Dalichand_Virchand_Shroff_And_Others_on_1_March_1968_1 0.8345
6 Bhoju_Mandal_vs_Debnath_Bhagat_on_14_November_1962_1 0.8341
7 State_Of_U_P_vs_Zahoor_Ahmad_Anr_on_8_August_1973_1 0.8332
8 Surja_vs_Hardeva_And_Ors_on_17_October_1968_1 0.8312
9 Mangal_Prasad_Dead_By_Lrs_And_Another_vs_Krishna_Kumar_Maheshwari_And_Others_on_14_November_1991_1 0.8311
10 Man_Kaur_Dead_By_Lrs_vs_Hartar_Singh_Sangha_on_5_October_2010_1 0.8297
